In [9]:
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
import os

load_dotenv()

pd.set_option('display.max_colwidth', None)

SECRET_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
SUPABASE_URL = os.getenv("SUPABASE_URL")

In [81]:
supabase_cli = create_client(
    SUPABASE_URL,
    SECRET_KEY
)

In [11]:
df = pd.read_csv("test.csv")

In [12]:
# ['Unnamed: 0', 'series_title', 'card_img', 'detailed_url', 'amount', 'collection_names']
print(df["series_title"])

0                                     Luminous Reverie Series
1                                         Angel's Poem Series
2                      Ethereal Realm: Mythical Beasts Series
3      Baby Molly Winter Fit Check Series-Vinyl Plush Pendant
4               Road Journal Series-Plush Doll Pendant Series
                                ...                          
101                           POP BEAN Macaron Dessert Series
102                                    Let's Checkmate Series
103                                    Crying for Love Series
104                 Baby Molly - My Huggable Discovery Series
105                                   Snuggle With You Series
Name: series_title, Length: 106, dtype: object


In [13]:
print(df["card_img"])

0                                               https://cdn.prod.website-files.com/65d92e68000f827cfefa05e5/69409ca923a8459a2d1c2b62_Luminous%20Reverie%20Series%20thumb.webp
1                                               https://cdn.prod.website-files.com/65d92e68000f827cfefa05e5/69409bbb3bbd0146006cded3_Angel%27s%20Poem%20Series%20tji%2Cb.webp
2                                      https://cdn.prod.website-files.com/65d92e68000f827cfefa05e5/694099d523805a52b6fc190f_Ethereal%20Realm%20Mythical%20Beasts%20thumb.webp
3      https://cdn.prod.website-files.com/65d92e68000f827cfefa05e5/693caf150a4aacdf21b50f53_Baby%20Molly%20Winter%20Fit%20Check%20Series-Vinyl%20Plush%20Pendant%20thumb.webp
4                            https://cdn.prod.website-files.com/65d92e68000f827cfefa05e5/693cad51ad7840c904c5ac60_Road%20Journal%20Series-Plush%20Doll%20Pendant%20thumb.webp
                                                                                        ...                                       

In [36]:
amounts = df["amount"]

def int_conversion(amount):
    numbers = []
    for part in amount.split(','):
        num = ''.join(filter(str.isdigit, part))
        if num:
            numbers.append(int(num))
    return numbers

test = amounts.apply(int_conversion)
with open("output.txt", 'w') as file:
    for amt in test:
        file.write(str(amt))
        file.write(" " + "\n")


In [47]:
series_amt_df = pd.DataFrame()

In [69]:
all_numbers = []

with open("series_amt.txt", 'r') as file:
    for line in file:
        numbers = []
        count_per_series = 0
        amount = line.strip()
        for part in amount.split(','):
            num = ''.join(filter(str.isdigit, part))
            if num:
                # numbers.append(int(num))
                count_per_series += int(num)
        # print(count_per_series)
        # all_numbers.append(numbers)
        all_numbers.append(count_per_series)

In [45]:
df_joined = pd.concat([df["series_title"], df["card_img"]], axis=1)

In [70]:
df_joined["totalItems"] = all_numbers

In [71]:
print(df_joined)

                                               series_title  \
0                                   Luminous Reverie Series   
1                                       Angel's Poem Series   
2                    Ethereal Realm: Mythical Beasts Series   
3    Baby Molly Winter Fit Check Series-Vinyl Plush Pendant   
4             Road Journal Series-Plush Doll Pendant Series   
..                                                      ...   
101                         POP BEAN Macaron Dessert Series   
102                                  Let's Checkmate Series   
103                                  Crying for Love Series   
104               Baby Molly - My Huggable Discovery Series   
105                                 Snuggle With You Series   

                                                                                                                                                                   card_img  \
0                                             https://cdn.prod.websi

In [74]:
df_joined = df_joined.rename(columns={"series_title": "name", "card_img": "imageUrl"})

In [100]:
df_joined['id'] = range(10, len(df) + 10) 

In [104]:
print(df_joined)

      id                                                    name description  \
0     10                                 Luminous Reverie Series               
1     11                                     Angel's Poem Series               
2     12                  Ethereal Realm: Mythical Beasts Series               
3     13  Baby Molly Winter Fit Check Series-Vinyl Plush Pendant               
4     14           Road Journal Series-Plush Doll Pendant Series               
..   ...                                                     ...         ...   
101  111                         POP BEAN Macaron Dessert Series               
102  112                                  Let's Checkmate Series               
103  113                                  Crying for Love Series               
104  114               Baby Molly - My Huggable Discovery Series               
105  115                                 Snuggle With You Series               

                                       

In [102]:
df_joined["description"] = " "

In [103]:
new_order = ["id", "name", "description", "imageUrl", "totalItems"]

# Reorder the DataFrame
df_joined = df_joined[new_order]

In [90]:
print(df_joined.dtypes)

name           object
description    object
imageUrl       object
totalItems      int64
dtype: object


In [105]:
df_joined.to_csv("series_data.csv")

In [76]:
data_to_insert = df_joined.to_dict(orient="records")

In [82]:
response = supabase_cli.table("Series").insert(data_to_insert).execute()

APIError: {'message': 'permission denied for schema public', 'code': '42501', 'hint': None, 'details': None}